CloudWatch, CloudTrail, and Config are AWS's core observability and governance services. CloudWatch monitors the performance of your resources and applications — metrics, logs, and alarms. CloudTrail records every API call made in your account — the who, what, when, and where of all AWS activity. Config tracks the configuration state of your resources over time — recording what changed and whether it complies with your policies. Together they answer three fundamental questions: how is my system performing, who did what, and is my infrastructure compliant?

## CloudWatch Metrics

A **metric** is a time-series of data points representing a measurable value — CPU utilisation, request count, latency, error rate. Metrics are the foundation of CloudWatch.

### Metric structure
- **Namespace**: logical grouping — `AWS/EC2`, `AWS/RDS`, `AWS/Lambda`, or custom namespaces
- **Metric name**: `CPUUtilization`, `NetworkIn`, `Duration`
- **Dimensions**: key-value pairs that identify the specific resource — `{InstanceId: i-abc123}`
- **Data points**: timestamp + value + unit (Percent, Bytes, Count, Seconds, etc.)
- **Resolution**: Standard (1-minute) or High Resolution (1-second for custom metrics)

### Standard metrics vs custom metrics

| | Standard metrics | Custom metrics |
|---|---|---|
| Source | AWS services (automatic) | Your application or CloudWatch Agent |
| Cost | Free | $0.30 per metric/month (first 10,000 free) |
| Examples | EC2 CPU, S3 bucket size, Lambda errors | Memory usage, disk space, app request count |
| EC2 coverage | CPU, Network, Disk I/O (hypervisor-level) | Memory, swap, disk space (need Agent inside OS) |

> **Important:** CloudWatch does **not** collect EC2 memory utilisation or disk space by default — these require the **CloudWatch Agent** installed on the instance.

### Metric retention
- < 60-second resolution: retained for 3 hours
- 1-minute: retained for 15 days
- 5-minute: retained for 63 days
- 1-hour: retained for 455 days (15 months)

## CloudWatch Alarms

Alarms watch a metric and trigger actions when it crosses a threshold.

### Alarm states
| State | Meaning |
|---|---|
| **OK** | Metric is within the threshold |
| **ALARM** | Metric has breached the threshold |
| **INSUFFICIENT_DATA** | Not enough data to evaluate (startup, gaps) |

### Alarm actions
- **SNS notification** — email, SMS, trigger Lambda
- **Auto Scaling action** — scale in or scale out an ASG
- **EC2 action** — stop, terminate, reboot, or recover an instance
- **Systems Manager action** — run an automation runbook

### Alarm evaluation
Configure the number of evaluation periods and the period duration:
```text
Threshold: CPU > 80%
Period: 300 seconds (5 minutes)
Evaluation periods: 3
Datapoints to alarm: 2 out of 3
→ ALARM if CPU > 80% in 2 of the last 3 five-minute periods
```

### Composite Alarms
Combine multiple alarms with AND/OR logic to reduce alert noise:
```text
ALARM if (HighCPU ALARM) AND (HighMemory ALARM)
  → Avoids false positives from a single metric spike
```

### Billing alarm
Create a billing alarm on the `AWS/Billing` metric `EstimatedCharges` in **us-east-1** (billing metrics are only available in us-east-1). Must enable billing alerts in account preferences first.

## CloudWatch Logs

CloudWatch Logs collects, stores, and queries log data from AWS services and your applications.

### Structure
```text
Log Group  (e.g., /aws/lambda/my-function)
  └── Log Stream  (e.g., 2026/04/19/[$LATEST]abc123)
         └── Log Events  (individual log lines with timestamp)
```

- **Log Group**: the primary container; set retention (1 day – 10 years, or never expire)
- **Log Stream**: sequence of events from a single source (one Lambda instance, one EC2 instance)
- **Log Events**: the actual log lines

### Sources
- Lambda — automatically sends logs to `/aws/lambda/<function-name>`
- ECS — container stdout/stderr
- API Gateway — access logs
- CloudTrail — trail logs
- VPC Flow Logs — network traffic
- CloudWatch Agent — EC2 OS logs and application logs
- Route 53 query logs

### Metric Filters
Extract metric data points from log text:
```text
Pattern: [ERROR]  →  increment ErrorCount metric by 1
```
Use metric filters to turn unstructured logs into CloudWatch metrics, then alarm on them.

### CloudWatch Logs Insights
Interactive query language for analysing log data:
```sql
fields @timestamp, @message
| filter @message like /ERROR/
| stats count(*) by bin(5m)
| sort @timestamp desc
| limit 20
```
Queries run across multiple log groups simultaneously. Results can be visualised as time-series charts or pie charts.

### Log encryption and export
- Logs encrypted at rest using KMS (CMK)
- Export logs to **S3** for long-term archiving (batch export, not real-time)
- **Subscription filter** streams logs in real-time to: Kinesis Data Streams, Kinesis Firehose, or Lambda

## CloudWatch Agent

The CloudWatch Agent runs inside EC2 instances (or on-premises servers) and sends:
- **System-level metrics**: memory utilisation, swap usage, disk space, disk I/O (not available from the hypervisor)
- **Log files**: application logs, OS logs, custom log files

```text
EC2 Instance
  ├── CloudWatch Agent ──▶ CloudWatch Metrics  (memory, disk space)
  └── CloudWatch Agent ──▶ CloudWatch Logs     (/var/log/app.log)
```

Configured via a JSON config file (or SSM Parameter Store for central management). Uses an IAM role attached to the EC2 instance with `CloudWatchAgentServerPolicy`.

### Container and Lambda Insights
- **Container Insights**: enhanced metrics for ECS and EKS — CPU, memory, disk, network per container; collects container logs
- **Lambda Insights**: function duration, memory usage, cold start count, error rates; requires a Lambda layer

## AWS CloudTrail

CloudTrail records every **API call** made in your AWS account — via the console, AWS CLI, SDKs, or other services. It is the audit log of all AWS activity.

### What CloudTrail records
Every event includes:
- **Who**: IAM user, role, or service that made the call
- **What**: API action (`RunInstances`, `PutObject`, `CreateBucket`)
- **When**: timestamp
- **Where**: source IP address, AWS region
- **Result**: success or error code

### Event types

| Type | Description | Default? | Cost |
|---|---|---|---|
| **Management events** | Control plane operations — create/delete/modify resources | Yes (free) | Free (first copy) |
| **Data events** | Object-level operations — S3 GetObject/PutObject, Lambda invocations, DynamoDB GetItem | No | Charged |
| **Insights events** | Anomalous API activity detection | No | Charged |

### Trails
- **Event history**: last 90 days of management events, viewable in the console — free, no setup
- **Trail**: delivers events to an S3 bucket (and optionally CloudWatch Logs) — required for retention beyond 90 days and for data events
- **Multi-region trail**: captures events from all regions into one S3 bucket — recommended default
- **Organisation trail**: captures events from all accounts in an AWS Organisation

### CloudTrail Insights
Detects **unusual API activity** using ML baselines:
- Abnormal spikes in `TerminateInstances`, `DeleteSecurityGroup`, `CreateAccessKey`
- Detects both write API anomalies and read API anomalies
- Insights events stored in S3 and delivered to CloudWatch Events

### CloudTrail integrity
- **Log file validation**: CloudTrail generates a digest file (SHA-256 hash) every hour. Use `aws cloudtrail validate-logs` to detect tampering or deletion.
- Protect trail S3 bucket: enable MFA Delete, use S3 Object Lock, restrict bucket policy

> **Exam tip:** "Who deleted the S3 bucket?", "which IAM user launched these instances?", "audit API activity" → CloudTrail.

## AWS Config

Config continuously records the **configuration state** of your AWS resources and evaluates them against compliance rules.

### What Config does
1. **Records** the configuration of supported resources (EC2, S3, RDS, IAM, VPC, SGs, etc.) at every change
2. **Stores** the configuration history in an S3 bucket
3. **Evaluates** resources against Config rules to determine compliance
4. **Notifies** via SNS on compliance changes
5. **Remediates** non-compliant resources via SSM Automation

### Config timeline
```text
Time ──────────────────────────────────────────────────────▶
       Config A      Config B      Config C      Config D
       (baseline)    (changed)     (changed)     (current)
           │             │             │
       [COMPLIANT]  [NON-COMPLIANT] [COMPLIANT]
```
You can see **what a resource looked like at any point in time** and whether it was compliant.

### Config Rules

| Type | Description | Examples |
|---|---|---|
| **AWS Managed rules** | Pre-built rules from AWS — 200+ available | `encrypted-volumes`, `s3-bucket-public-read-prohibited`, `restricted-ssh` |
| **Custom rules** | Lambda function evaluates compliance | Any custom business logic |
| **Service-linked rules** | Created by AWS services (e.g., Security Hub) | — |

### Config Aggregator
Aggregate configuration and compliance data from **multiple accounts and regions** into a single account. Requires AWS Organisation or individual account authorisation.

### Remediation
Config can trigger **automatic remediation** for non-compliant resources:
- Associate a rule with an SSM Automation document
- On rule violation: SSM runs the document (e.g., enable S3 bucket encryption, revoke an overly permissive security group)
- Can require manual approval before remediation

## CloudWatch vs CloudTrail vs Config

| Question | Service |
|---|---|
| Is my CPU high? Is my Lambda erroring? | **CloudWatch** — performance monitoring |
| Who deleted that S3 bucket? Who changed this IAM policy? | **CloudTrail** — API audit log |
| What did my security group look like last Tuesday? Was it compliant? | **Config** — resource configuration history |
| Alert me when RDS CPU stays above 80% for 10 minutes | **CloudWatch** alarm |
| Alert me when someone calls `DeleteTrail` | **CloudTrail** → CloudWatch metric filter → alarm |
| Alert me when an S3 bucket becomes publicly accessible | **Config** rule → SNS |
| Automatically fix a non-compliant security group | **Config** remediation → SSM |

```text
CloudWatch  =  Performance & health     (metrics, logs, alarms)
CloudTrail  =  Who did what             (API activity audit)
Config      =  What changed & compliant (configuration history + rules)
```

## Working with CloudWatch, CloudTrail, and Config via boto3

In [ ]:
import boto3
import json
from datetime import datetime, timedelta

cw         = boto3.client('cloudwatch',  region_name='us-east-1')
logs       = boto3.client('logs',        region_name='us-east-1')
cloudtrail = boto3.client('cloudtrail',  region_name='us-east-1')
config     = boto3.client('config',      region_name='us-east-1')

In [ ]:
# Publish a custom metric
cw.put_metric_data(
    Namespace='MyApp/Business',
    MetricData=[
        {
            'MetricName': 'OrdersProcessed',
            'Value': 142,
            'Unit': 'Count',
            'Dimensions': [
                {'Name': 'Environment', 'Value': 'production'},
                {'Name': 'Region',      'Value': 'us-east-1'}
            ]
        }
    ]
)
print("Custom metric published")

In [ ]:
# Create a CloudWatch alarm
cw.put_metric_alarm(
    AlarmName='HighCPU-prod-web',
    AlarmDescription='CPU > 80% for 2 consecutive 5-minute periods',
    Namespace='AWS/EC2',
    MetricName='CPUUtilization',
    Dimensions=[{'Name': 'AutoScalingGroupName', 'Value': 'prod-web-asg'}],
    Statistic='Average',
    Period=300,                  # 5 minutes
    EvaluationPeriods=2,
    DatapointsToAlarm=2,
    Threshold=80.0,
    ComparisonOperator='GreaterThanThreshold',
    TreatMissingData='notBreaching',
    AlarmActions=[
        'arn:aws:sns:us-east-1:123456789012:ops-alerts',          # notify ops
        'arn:aws:autoscaling:us-east-1:123456789012:scalingPolicy:abc:autoScalingGroupName/prod-web-asg:policyName/scale-out'  # scale out
    ]
)
print("Alarm created")

In [ ]:
# Run a CloudWatch Logs Insights query
response = logs.start_query(
    logGroupName='/aws/lambda/my-function',
    startTime=int((datetime.now() - timedelta(hours=1)).timestamp()),
    endTime=int(datetime.now().timestamp()),
    queryString="""
        fields @timestamp, @message
        | filter @message like /ERROR/
        | stats count(*) as errorCount by bin(5m)
        | sort @timestamp desc
    """
)
query_id = response['queryId']
print(f"Query started: {query_id}")

# Poll for results
import time
while True:
    result = logs.get_query_results(queryId=query_id)
    if result['status'] in ('Complete', 'Failed', 'Cancelled'):
        break
    time.sleep(1)

for row in result['results'][:5]:
    fields = {f['field']: f['value'] for f in row}
    print(f"Time: {fields.get('@timestamp', '')} | Errors: {fields.get('errorCount', '')}")

In [ ]:
# Create a multi-region CloudTrail trail
cloudtrail.create_trail(
    Name='org-audit-trail',
    S3BucketName='my-cloudtrail-logs',
    IsMultiRegionTrail=True,         # capture events from all regions
    IncludeGlobalServiceEvents=True, # IAM, STS, CloudFront (global services)
    EnableLogFileValidation=True,    # SHA-256 digest for tamper detection
    CloudWatchLogsLogGroupArn='arn:aws:logs:us-east-1:123456789012:log-group:CloudTrail',
    CloudWatchLogsRoleArn='arn:aws:iam::123456789012:role/CloudTrailCWRole',
    KMSKeyId='alias/cloudtrail-key'
)
cloudtrail.start_logging(Name='org-audit-trail')
print("Multi-region CloudTrail trail created and started")

In [ ]:
# Look up recent CloudTrail events (event history — last 90 days)
events = cloudtrail.lookup_events(
    LookupAttributes=[{
        'AttributeKey':   'EventName',
        'AttributeValue': 'DeleteBucket'
    }],
    StartTime=datetime.now() - timedelta(days=7),
    EndTime=datetime.now(),
    MaxResults=5
)
for event in events['Events']:
    print(f"Time: {event['EventTime']} | User: {event.get('Username', 'N/A')} | "
          f"Resource: {event['Resources'][0]['ResourceName'] if event.get('Resources') else 'N/A'}")

In [ ]:
# Enable AWS Config recording
config.put_configuration_recorder(
    ConfigurationRecorder={
        'name': 'default',
        'roleARN': 'arn:aws:iam::123456789012:role/ConfigRole',
        'recordingGroup': {
            'allSupported': True,
            'includeGlobalResourceTypes': True  # include IAM resources
        }
    }
)
config.put_delivery_channel(
    DeliveryChannel={
        'name': 'default',
        's3BucketName': 'my-config-bucket',
        'snsTopicARN': 'arn:aws:sns:us-east-1:123456789012:config-alerts',
        'configSnapshotDeliveryProperties': {'deliveryFrequency': 'TwentyFour_Hours'}
    }
)
config.start_configuration_recorder(ConfigurationRecorderName='default')
print("Config recording started")

# Add a managed Config rule — flag unencrypted EBS volumes
config.put_config_rule(
    ConfigRule={
        'ConfigRuleName': 'encrypted-volumes',
        'Source': {
            'Owner': 'AWS',
            'SourceIdentifier': 'ENCRYPTED_VOLUMES'
        },
        'Scope': {
            'ComplianceResourceTypes': ['AWS::EC2::Volume']
        }
    }
)
print("Config rule added: encrypted-volumes")

In [ ]:
# Check compliance status across all Config rules
compliance = config.describe_compliance_by_config_rule()
for rule in compliance['ComplianceByConfigRules']:
    status = rule['Compliance']['ComplianceType']
    print(f"Rule: {rule['ConfigRuleName']:<40} Status: {status}")

## Summary

| Concept | Key Takeaway |
|---|---|
| CloudWatch Metrics | Time-series data; namespace + metric name + dimensions; standard (free) vs custom |
| EC2 memory/disk | NOT in standard metrics — requires CloudWatch Agent inside the OS |
| High Resolution metrics | 1-second granularity for custom metrics (vs 1-minute standard) |
| CloudWatch Alarms | Watch metric vs threshold; states: OK / ALARM / INSUFFICIENT_DATA |
| Alarm actions | SNS notification, Auto Scaling, EC2 action, SSM runbook |
| Composite Alarms | AND/OR logic across multiple alarms — reduce alert noise |
| CloudWatch Logs | Log Groups → Log Streams → Log Events; set retention per group |
| Metric Filters | Extract metrics from log text (count ERRORs → alarm on error rate) |
| Logs Insights | Interactive SQL-like query language across log groups |
| Log export | S3 (batch); Kinesis / Firehose / Lambda (real-time subscription filter) |
| CloudWatch Agent | EC2 memory, disk space, application log files; uses IAM role |
| CloudTrail | API audit log: who, what, when, where for every AWS API call |
| Management events | Default; create/modify/delete resources; free for first copy |
| Data events | S3 object operations, Lambda invocations — opt-in, charged |
| CloudTrail Insights | ML-based anomaly detection on API write/read volumes |
| Multi-region trail | Capture all regions into one S3 bucket; recommended default |
| Log file validation | SHA-256 digest every hour — detect trail tampering or deletion |
| AWS Config | Records resource configuration history; evaluates compliance rules |
| Config rules | AWS managed (200+) or custom Lambda — evaluate resource compliance |
| Config Aggregator | Multi-account / multi-region compliance view from one account |
| Config remediation | Auto-remediate non-compliant resources via SSM Automation |
| CW vs CT vs Config | CW: performance; CloudTrail: who did what; Config: what changed + compliant? |